# 📡 TOP 500 Calls Heard in Poland (RBN)
Ten notebook pobiera dane z Reverse Beacon Network, przetwarza pliki ZIP i generuje listę najczęściej słyszanych stacji CW w Polsce.

In [ ]:
!pip install tqdm

In [ ]:
import os, sys, time, zipfile
from datetime import datetime, timedelta
from collections import Counter
from urllib.request import urlopen, Request
from urllib.error import URLError, HTTPError
import pandas as pd
from tqdm.notebook import tqdm


: 

In [ ]:
BASE_URL = "https://data.reversebeacon.net/rbn_history/{YMD}.zip"
DATA_DIR = "data"
OUT_TXT = "morse_runner_calls.txt"
OUT_CSV = "top_calls_sp_cw.csv"

def daterange(d1, d2):
    d = d1
    while d <= d2:
        yield d
        d += timedelta(days=1)

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def url_for(d):
    return BASE_URL.format(YMD=d.strftime("%Y%m%d"))

def download_zip(dst, url, retries=3, timeout=60):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return True
    for att in range(1, retries+1):
        try:
            req = Request(url, headers={"User-Agent": "rbn-fetch/1.0"})
            with urlopen(req, timeout=timeout) as r, open(dst, "wb") as f:
                f.write(r.read())
            if os.path.getsize(dst) == 0:
                raise IOError("Empty file")
            return True
        except (HTTPError, URLError, IOError) as e:
            if att < retries:
                time.sleep(2*att)
            else:
                print(f"[WARN] {url} -> {e}", file=sys.stderr)
                return False

def process_zip(path) -> Counter:
    c = Counter()
    try:
        with zipfile.ZipFile(path) as zf:
            names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
            if not names:
                return c
            with zf.open(names[0]) as fh:
                try:
                    df = pd.read_csv(
                        fh, header=None,
                        names=["poster","poster_country_prefix","poster_continent","freq_khz","band",
                               "dx","dx_country_prefix","dx_continent","cq","snr_db","datetime_utc",
                               "wpm","mode","date_compact","epoch"],
                        usecols=["poster_country_prefix","dx","mode"],
                        dtype=str,
                        low_memory=False
                    )
                except:
                    fh.seek(0)
                    df = pd.read_csv(
                        fh, header=None,
                        names=["poster","poster_country_prefix","poster_continent","freq_khz","band",
                               "dx","dx_country_prefix","dx_continent","cq","snr_db","datetime_utc",
                               "wpm","mode"],
                        usecols=["poster_country_prefix","dx","mode"],
                        dtype=str,
                        low_memory=False
                    )
                sub = df[(df["mode"] == "CW") & (df["poster_country_prefix"] == "SP")]
                if not sub.empty:
                    c.update(sub["dx"].value_counts().to_dict())
    except:
        print(f"[WARN] Problem with ZIP: {path}", file=sys.stderr)
    return c


In [ ]:
from datetime import datetime

date_from = "2024-08-01"
date_to   = "2024-08-05"   # Możesz zmienić zakres dat
topn = 500

d1 = datetime.strptime(date_from, "%Y-%m-%d")
d2 = datetime.strptime(date_to, "%Y-%m-%d")
ensure_dir(DATA_DIR)
total = Counter()

print(f"Pobieranie danych z {d1.date()} do {d2.date()}...")
for d in tqdm(list(daterange(d1, d2)), desc="Pobieranie i przetwarzanie"):
    url  = url_for(d)
    dst  = os.path.join(DATA_DIR, f"{d.strftime('%Y%m%d')}.zip")
    if not download_zip(dst, url):
        continue
    total.update(process_zip(dst))

top = total.most_common(topn)

import pandas as pd
df = pd.DataFrame(top, columns=["callsign", "count"])
df.to_csv(OUT_CSV, index=False)
df.head(20)


✅ Gotowe! Plik CSV zapisany jako `top_calls_sp_cw.csv`. Możesz go pobrać lub otworzyć bezpośrednio z Colab.